In [1]:
import requests
import json
import pandas as pd
import time

The initial trial in collecting data from jolpi's F1 api, where i'm solely checking for data in the current (2026) season. Given that it's only been 5 races (not including the past race in Monaco), data from past seasons from 2022 will be used to predict the standings of this season.

In [31]:
test_url = "https://api.jolpi.ca/ergast/f1/2026/results/?limit=1000"
response = requests.get(test_url)

print("Status code:", response.status_code)
print("Content-type:", response.headers.get("Content-type"))

Status code: 200
Content-type: application/json


In [32]:
data = response.json()
races = data['MRData']['RaceTable']['Races']

In [33]:
#convert json response to df
pd.json_normalize(races)

,season,round,url,raceName,date,time,Results,Circuit.circuitId,Circuit.url,Circuit.circuitName,Circuit.Location.lat,Circuit.Location.long,Circuit.Location.locality,Circuit.Location.country
0,2026,1,https://en.wikipedia.org/wiki/2026_Australian_...,Australian Grand Prix,2026-03-08,04:00:00Z,"[{'number': '63', 'position': '1', 'positionTe...",albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
1,2026,2,https://en.wikipedia.org/wiki/2026_Chinese_Gra...,Chinese Grand Prix,2026-03-15,07:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",shanghai,https://en.wikipedia.org/wiki/Shanghai_Interna...,Shanghai International Circuit,31.3389,121.22,Shanghai,China
2,2026,3,https://en.wikipedia.org/wiki/2026_Japanese_Gr...,Japanese Grand Prix,2026-03-29,05:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",suzuka,https://en.wikipedia.org/wiki/Suzuka_Internati...,Suzuka Circuit,34.8431,136.541,Suzuka,Japan
3,2026,4,https://en.wikipedia.org/wiki/2026_Miami_Grand...,Miami Grand Prix,2026-05-03,20:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",miami,https://en.wikipedia.org/wiki/Miami_Internatio...,Miami International Autodrome,25.9581,-80.2389,Miami,USA
4,2026,5,https://en.wikipedia.org/wiki/2026_Canadian_Gr...,Canadian Grand Prix,2026-05-24,20:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",villeneuve,https://en.wikipedia.org/wiki/Circuit_Gilles_V...,Circuit Gilles Villeneuve,45.5,-73.5228,Montreal,Canada


In [34]:
test_df = pd.json_normalize(
    races,
    record_path='Results',
    meta = ['season', 'round', 'raceName', 'date', ['Circuit', 'circuitId']]
)
test_df

,number,position,positionText,points,grid,laps,status,Driver.driverId,Driver.permanentNumber,Driver.code,...,Time.millis,Time.time,FastestLap.rank,FastestLap.lap,FastestLap.Time.time,season,round,raceName,date,Circuit.circuitId
0,63,1,1,25,1,58,Finished,russell,63,RUS,...,4986801,1:23:06.801,6,21,1:22.670,2026,1,Australian Grand Prix,2026-03-08,albert_park
1,12,2,2,18,2,58,Finished,antonelli,12,ANT,...,4989775,+2.974,3,57,1:22.417,2026,1,Australian Grand Prix,2026-03-08,albert_park
2,16,3,3,15,4,58,Finished,leclerc,16,LEC,...,5002320,+15.519,5,38,1:22.579,2026,1,Australian Grand Prix,2026-03-08,albert_park
3,44,4,4,12,7,58,Finished,hamilton,44,HAM,...,5002945,+16.144,4,55,1:22.423,2026,1,Australian Grand Prix,2026-03-08,albert_park
4,1,5,5,10,6,58,Finished,norris,1,NOR,...,5038542,+51.741,2,53,1:22.358,2026,1,Australian Grand Prix,2026-03-08,albert_park
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,10,8,8,4,14,67,Lapped,gasly,10,GAS,...,5330330,+34.572,6,67,1:15.390,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
96,55,9,9,2,15,67,Lapped,sainz,55,SAI,...,5353772,+58.014,12,65,1:15.852,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
97,87,10,10,1,16,67,Lapped,bearman,87,BEA,...,5354807,+59.049,13,64,1:16.002,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
98,81,11,11,0,4,66,Lapped,piastri,81,PIA,...,5308457,+12.699,7,61,1:15.456,2026,5,Canadian Grand Prix,2026-05-24,villeneuve


Data being collected will start from the 2022 season upto the 5 rounds that have occured in the F1 season in 2026 so far. This is to reflect the ground effect era of cars that began in 2022, while also using the limited data from the 2026 season to take into account the new regulations that began this season

In [4]:
def get_data(season, page_size=100, pause=0.2):
    races = []
    offset = 0
    #pulling data from multiple pages since initial get only provides 5 races worth of data
    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{season}/results/?limit={page_size}&offset={offset}"
        response = requests.get(url).json()['MRData']
        races.extend(response['RaceTable']['Races'])
        #number of rows in the entire season
        total = int(response['total'])
        #move on to the next page
        offset += page_size
        
        if offset >= total:
            break
    
        time.sleep(pause)
        
    return pd.json_normalize(races, record_path='Results', meta = ['season', 'round', 'raceName', 'date', ['Circuit', 'circuitId']]
)

In [6]:
seasons = [2022, 2023, 2024, 2025, 2026]

#forming the dataframe
df = pd.concat([get_data(season) for season in seasons], ignore_index=True)
df

,number,position,positionText,points,grid,laps,status,Driver.driverId,Driver.permanentNumber,Driver.code,...,FastestLap.rank,FastestLap.lap,FastestLap.Time.time,FastestLap.AverageSpeed.units,FastestLap.AverageSpeed.speed,season,round,raceName,date,Circuit.circuitId
0,16,1,1,26,1,57,Finished,leclerc,16,LEC,...,1,51,1:34.570,kph,206.018,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
1,55,2,2,18,3,57,Finished,sainz,55,SAI,...,3,52,1:35.740,kph,203.501,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
2,44,3,3,15,5,57,Finished,hamilton,44,HAM,...,5,53,1:36.228,kph,202.469,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
3,63,4,4,12,9,57,Finished,russell,63,RUS,...,6,56,1:36.302,kph,202.313,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
4,20,5,5,10,7,57,Finished,kevin_magnussen,20,MAG,...,8,53,1:36.623,kph,201.641,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1965,18,18,R,0,22,56,Retired,stroll,18,STR,...,20,36,1:18.845,NaN,NaN,2026,6,Monaco Grand Prix,2026-06-07,monaco
1966,1,19,R,0,8,43,Retired,norris,1,NOR,...,17,34,1:17.670,NaN,NaN,2026,6,Monaco Grand Prix,2026-06-07,monaco
1967,87,20,R,0,19,27,Retired,bearman,87,BEA,...,19,22,1:18.475,NaN,NaN,2026,6,Monaco Grand Prix,2026-06-07,monaco
1968,77,21,R,0,20,15,Retired,bottas,77,BOT,...,21,8,1:20.494,NaN,NaN,2026,6,Monaco Grand Prix,2026-06-07,monaco


In [7]:
df.shape

(1970, 31)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1970 entries, 0 to 1969
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   number                         1970 non-null   object
 1   position                       1970 non-null   object
 2   positionText                   1970 non-null   object
 3   points                         1970 non-null   object
 4   grid                           1970 non-null   object
 5   laps                           1970 non-null   object
 6   status                         1970 non-null   object
 7   Driver.driverId                1970 non-null   object
 8   Driver.permanentNumber         1970 non-null   object
 9   Driver.code                    1970 non-null   object
 10  Driver.url                     1970 non-null   object
 11  Driver.givenName               1970 non-null   object
 12  Driver.familyName              1970 non-null   object
 13  Dri

In [9]:
df.nunique()

number                             35
position                           22
positionText                       23
points                             19
grid                               23
laps                               76
status                             33
Driver.driverId                    32
Driver.permanentNumber             29
Driver.code                        32
Driver.url                         32
Driver.givenName                   32
Driver.familyName                  32
Driver.dateOfBirth                 32
Driver.nationality                 19
Constructor.constructorId          14
Constructor.url                    14
Constructor.name                   14
Constructor.nationality             7
Time.millis                      1624
Time.time                        1585
FastestLap.rank                    22
FastestLap.lap                     77
FastestLap.Time.time             1856
FastestLap.AverageSpeed.units       1
FastestLap.AverageSpeed.speed    1271
season      

In [11]:
#save the data as a csv for cleaning and processing
df.to_csv('results.csv', index=False)